In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Synthetic Landing Event (for testing only)
# MAGIC Inserts one fake "on_ground" row for a flight that's already showing
# MAGIC "in_air" in the landing table, to artificially trigger the
# MAGIC on_ground/in_air transition detection in flight_status_events —
# MAGIC without waiting for a real flight to actually land.
# MAGIC
# MAGIC Fill in FLIGHT_NUMBER and FLIGHT_DATE below to match the real
# MAGIC watched flight you've already been polling.

# COMMAND ----------

dbutils.widgets.text("catalog", "bootcamp_students")
dbutils.widgets.text("schema", "madgula_sirisha_capstone")
dbutils.widgets.text("landing_table", "adsb_lol_flight_landing")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
landing_table = dbutils.widgets.get("landing_table")
full_table_name = f"{catalog}.{schema}.{landing_table}"

# COMMAND ----------

FLIGHT_NUMBER = "DAL1830"        # <-- set to your real watched flight_number
FLIGHT_DATE = "2026-09-24"      # <-- set to your real watched flight_date (YYYY-MM-DD)

# COMMAND ----------

import json
from datetime import datetime, timezone, date

from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, BooleanType, TimestampType, DateType

LANDING_ROW_SCHEMA = StructType([
    StructField("ingested_at", TimestampType(), nullable=False),
    StructField("flight_number", StringType(), nullable=False),
    StructField("flight_date", DateType(), nullable=False),
    StructField("found", BooleanType(), nullable=False),
    StructField("raw_json", StringType(), nullable=True),
    StructField("poll_error", StringType(), nullable=True),
])

# alt_baro = "ground" (the literal string, matching real adsb.lol format)
# is what makes the streaming pipeline classify this as status="on_ground".
synthetic_aircraft = {
    "hex": "a9cee9",
    "flight": FLIGHT_NUMBER,
    "alt_baro": "ground",
    "gs": 0.0,
    "track": 0.0,
    "lat": 33.6407,
    "lon": -84.4277,
    "squawk": "1200",
    "seen": 0.5,
    "seen_pos": 0.5,
}

synthetic_row = Row(
    ingested_at=datetime.now(timezone.utc),
    flight_number=FLIGHT_NUMBER,
    flight_date=date.fromisoformat(FLIGHT_DATE),
    found=True,
    raw_json=json.dumps(synthetic_aircraft),
    poll_error=None,
)

df = spark.createDataFrame([synthetic_row], schema=LANDING_ROW_SCHEMA)
df.write.mode("append").saveAsTable(full_table_name)

print(f"Inserted synthetic on_ground row for {FLIGHT_NUMBER} on {FLIGHT_DATE} into {full_table_name}")
print("Watch flight_status_events for a 'landing' event within the next micro-batch cycle.")